In [1]:

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector

# --------------------------------------------------
# STEP 1: Define the target matrix B
# --------------------------------------------------

X = np.array([[0, 1],
              [1, 0]], dtype=complex)

Z = np.array([[1, 0],
              [0, -1]], dtype=complex)

B = (1/3)*X + (2/3)*Z

print("Target matrix B:")
print(np.round(B, 6))

# --------------------------------------------------
# STEP 2: Calculate the PREPARE rotation angle
# --------------------------------------------------

theta = 2 * np.arccos(np.sqrt(1/3))

print("\nTheta (radians):", theta)
print("Theta (degrees):", np.degrees(theta))


# --------------------------------------------------
# STEP 3: Construct the LCU circuit
# q0 = ancilla, q1 = system
# --------------------------------------------------

qc = QuantumCircuit(2)

# PREPARE
qc.ry(theta, 0)

# SELECT:
# Apply X when ancilla q0 = 0
qc.x(0)
qc.cx(0, 1)
qc.x(0)

# Apply Z when ancilla q0 = 1
qc.cz(0, 1)

# PREPARE dagger
qc.ry(-theta, 0)

print("\nQuantum circuit:")
print(qc.draw(fold=-1))

# --------------------------------------------------
# STEP 4: Extract the full 4x4 unitary
# --------------------------------------------------

U = Operator(qc).data

print("\nFull unitary U:")
print(np.round(U, 6))

# --------------------------------------------------
# STEP 5: Extract the ancilla-|0> block
# --------------------------------------------------

# Qiskit basis order: |q1 q0>
# q0 is the ancilla.
# Ancilla = 0 corresponds to indices 0 and 2.

indices = [0, 2]

B_extracted = U[np.ix_(indices, indices)]

print("\nExtracted block:")
print(np.round(B_extracted, 6))


# --------------------------------------------------
# STEP 6: Verify block encoding
# --------------------------------------------------

print("\nDoes extracted block equal B?")
print(np.allclose(B_extracted, B))

print("\nIs the full matrix U unitary?")
print(np.allclose(U.conj().T @ U, np.eye(4)))


# --------------------------------------------------
# STEP 7: Verify using an input state |0>
# --------------------------------------------------

initial_state = Statevector.from_label("00")
final_state = initial_state.evolve(qc)

amps = final_state.data

branch_0 = np.array([amps[0], amps[2]])

classical_result = B @ np.array([1, 0])

print("\nQuantum ancilla-|0> branch:")
print(np.round(branch_0, 6))

print("\nClassical B|0>:")
print(np.round(classical_result, 6))

print("\nSuccess probability:")
print(np.sum(np.abs(branch_0)**2))


Target matrix B:
[[ 0.666667+0.j  0.333333+0.j]
 [ 0.333333+0.j -0.666667+0.j]]

Theta (radians): 1.9106332362490186
Theta (degrees): 109.47122063449069

Quantum circuit:
     ┌────────────┐┌───┐     ┌───┐   ┌─────────────┐
q_0: ┤ Ry(1.9106) ├┤ X ├──■──┤ X ├─■─┤ Ry(-1.9106) ├
     └────────────┘└───┘┌─┴─┐└───┘ │ └─────────────┘
q_1: ───────────────────┤ X ├──────■────────────────
                        └───┘                       

Full unitary U:
[[ 0.666667+0.j  0.471405+0.j  0.333333+0.j -0.471405+0.j]
 [ 0.471405+0.j  0.333333+0.j -0.471405+0.j  0.666667+0.j]
 [ 0.333333+0.j -0.471405+0.j -0.666667+0.j -0.471405+0.j]
 [-0.471405+0.j  0.666667+0.j -0.471405+0.j -0.333333+0.j]]

Extracted block:
[[ 0.666667+0.j  0.333333+0.j]
 [ 0.333333+0.j -0.666667+0.j]]

Does extracted block equal B?
True

Is the full matrix U unitary?
True

Quantum ancilla-|0> branch:
[0.666667+0.j 0.333333+0.j]

Classical B|0>:
[0.666667+0.j 0.333333+0.j]

Success probability:
0.5555555555555556
